# 06 节奏分析：起音检测、tempogram 与节拍跟踪

内容脉络：
1. **起音检测**：谱通量 + 峰值检测，估计声音事件开始的时刻
2. **tempogram**：局部自相关得到的速度强度图
3. **节拍跟踪**：动态规划从起音证据中提取周期性拍点
4. **Beat This!**（可选）：学习型方法对比

## 1. 环境自检与配置

In [ ]:
import sys
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from IPython import display as ipydisplay
from pathlib import Path
import warnings

import librosa
import librosa.display

SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_AUDIO_DIR = BASE_DIR / "CODE" / "chapter05" / "output_audio"
OUTPUT_FIG_DIR.mkdir(exist_ok=True)
OUTPUT_AUDIO_DIR.mkdir(exist_ok=True)

print(f"librosa={librosa.__version__}")

In [ ]:
# 依赖只做状态检查；Notebook 不在执行过程中自动安装软件包。
try:
    import beat_this
    print(f"Beat This!：{getattr(beat_this, '__version__', '已安装')}")
except ImportError:
    print("Beat This! 未安装；可在运行前由用户显式执行：pip install beat-this")


## 2. 加载音频

选择两段项目内录音：一段歌曲切片用于节拍跟踪，一段打击乐切片用于观察起音检测。文件名和听感不能替代节拍或风格标注，因此不据此把第二段归为“散板”或“自由节奏”。


In [ ]:
# 加载 10 秒片段
song_samples, sr = librosa.load(
    DATASET_DIR / "xiaohetang_full.wav", sr=SAMPLE_RATE, mono=True, offset=30.0, duration=10.0
)
perc_samples, sr = librosa.load(
    DATASET_DIR / "orch_perc.wav", sr=SAMPLE_RATE, mono=True, offset=2.0, duration=5.0
)

print(f"歌曲片段：{len(song_samples)} samples ({len(song_samples)/sr:.2f} s)")
print(f"打击乐片段：{len(perc_samples)} samples ({len(perc_samples)/sr:.2f} s)")

# 播放
display(ipydisplay.Audio(song_samples, rate=SAMPLE_RATE))

## 3. 起音检测

**起音（onset）** 是可感知声学事件开始或发生明显变化的时刻，例如鼓击、音符起音或某些和弦切换。歌词字头有时会产生起音证据，但两者并非一一对应。

`librosa.onset.onset_detect` 默认先调用 `onset_strength`。在 librosa 0.11 的默认路径中，后者从 dB-scaled mel 谱计算参考谱的正向差分，再沿频带聚合成起音强度包络；`max_size=1` 时频率方向的局部最大滤波关闭，调大后才启用颤音抑制参考谱。因此它不是简单的线性幅度谱逐 bin 相邻帧相减。包络峰越突出，越可能被后续峰值挑选判为起音。

In [ ]:
# 计算起音帧和起音强度曲线
hop_length = 512

# 歌曲
onset_frames_song = librosa.onset.onset_detect(y=song_samples, sr=SAMPLE_RATE, hop_length=hop_length)
onset_times_song = librosa.frames_to_time(onset_frames_song, sr=SAMPLE_RATE, hop_length=hop_length)
onset_env_song = librosa.onset.onset_strength(y=song_samples, sr=SAMPLE_RATE, hop_length=hop_length)
t_env_song = librosa.frames_to_time(np.arange(len(onset_env_song)), sr=SAMPLE_RATE, hop_length=hop_length)

# 打击乐
onset_frames_perc = librosa.onset.onset_detect(y=perc_samples, sr=SAMPLE_RATE, hop_length=hop_length)
onset_times_perc = librosa.frames_to_time(onset_frames_perc, sr=SAMPLE_RATE, hop_length=hop_length)
onset_env_perc = librosa.onset.onset_strength(y=perc_samples, sr=SAMPLE_RATE, hop_length=hop_length)
t_env_perc = librosa.frames_to_time(np.arange(len(onset_env_perc)), sr=SAMPLE_RATE, hop_length=hop_length)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

# 歌曲
axes[0].plot(t_env_song, onset_env_song, label="默认 mel 谱通量包络", color="0.1")
axes[0].vlines(onset_times_song, 0, np.max(onset_env_song), color="red", ls="--", alpha=0.7, label="检测到的起音")
axes[0].set_title("起音检测：歌曲切片")
axes[0].set_xlabel("时间 (s)")
axes[0].set_ylabel("起音强度")
axes[0].legend(loc="upper right")

# 打击乐
axes[1].plot(t_env_perc, onset_env_perc, label="默认 mel 谱通量包络", color="0.1")
axes[1].vlines(onset_times_perc, 0, np.max(onset_env_perc), color="red", ls="--", alpha=0.7, label="检测到的起音")
axes[1].set_title("起音检测：乐队打击乐")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylabel("起音强度")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "onset_detection.png", dpi=600, bbox_inches="tight")
plt.show()

print(f"歌曲检测到 {len(onset_times_song)} 个起音")
print(f"打击乐检测到 {len(onset_times_perc)} 个起音")

## 4. tempogram

`librosa.feature.tempogram` 计算局部起音强度包络的**短时自相关**，纵轴是延迟（显示时可换算为 BPM）。它不是“先自相关、再做傅里叶变换”；基于 ODF 的傅里叶版本是另一个函数 `librosa.feature.fourier_tempogram`。

稳定周期可能形成水平亮带，但半速/倍速候选通常会同时出现，亮带本身不是人工节拍真值。


In [ ]:
# 计算歌曲切片的局部自相关 tempogram
tempogram = librosa.feature.tempogram(
    y=song_samples, sr=SAMPLE_RATE,
    hop_length=hop_length, win_length=384
)

fig, ax = plt.subplots(figsize=(12, 4))
img = librosa.display.specshow(
    tempogram, sr=SAMPLE_RATE, hop_length=hop_length,
    x_axis="time", y_axis="tempo", ax=ax, cmap="Greys_r"
)
ax.set_ylim(60, 200)
ax.set_title("局部自相关 tempogram（歌曲切片）")
ax.set_xlabel("时间 (s)")
fig.colorbar(img, ax=ax)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "tempogram.png", dpi=600, bbox_inches="tight")
plt.show()


## 5. 节拍跟踪

`librosa.beat.beat_track` 从起音强度包络估计速度证据，再用动态规划选择节拍序列。`start_bpm` 是速度先验的中心，不是 DP 的初始状态；`tightness` 控制相邻拍间隔相对目标周期的对数惩罚。半速/倍速选择还取决于当前 ODF 的峰值与周期证据，不能只归因于这两个参数。


In [ ]:
# 两段切片分别运行同一节拍跟踪器；输出不等于真值
tempo_song, beat_frames_song = librosa.beat.beat_track(
    y=song_samples, sr=SAMPLE_RATE, hop_length=hop_length
)
beat_times_song = librosa.frames_to_time(
    beat_frames_song, sr=SAMPLE_RATE, hop_length=hop_length
)

tempo_perc, beat_frames_perc = librosa.beat.beat_track(
    y=perc_samples, sr=SAMPLE_RATE, hop_length=hop_length
)
beat_times_perc = librosa.frames_to_time(
    beat_frames_perc, sr=SAMPLE_RATE, hop_length=hop_length
)

tempo_song_value = float(np.asarray(tempo_song).reshape(-1)[0])
tempo_perc_value = float(np.asarray(tempo_perc).reshape(-1)[0])
print(f"歌曲切片：估计速度 ≈ {tempo_song_value:.1f} BPM，{len(beat_times_song)} 个节拍")
print(f"打击乐切片：估计速度 ≈ {tempo_perc_value:.1f} BPM，{len(beat_times_perc)} 个节拍")

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
librosa.display.waveshow(song_samples, sr=SAMPLE_RATE, ax=axes[0], alpha=0.6, color="0.3")
axes[0].vlines(beat_times_song, -1, 1, color="0.15", ls="--", lw=1.2, label="估计节拍")
axes[0].set_title(f"歌曲切片（估计速度 ≈ {tempo_song_value:.1f} BPM）")
axes[0].set_xlabel("时间 (s)")
axes[0].legend()

librosa.display.waveshow(perc_samples, sr=SAMPLE_RATE, ax=axes[1], alpha=0.6, color="0.3")
axes[1].vlines(beat_times_perc, -1, 1, color="0.15", ls="--", lw=1.2, label="估计节拍")
axes[1].set_title(f"打击乐切片（估计速度 ≈ {tempo_perc_value:.1f} BPM）")
axes[1].set_xlabel("时间 (s)")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "beat_tracking.png", dpi=600, bbox_inches="tight")
plt.show()


**观察口径**：两段切片都会得到周期网格，但本实验没有人工节拍真值。因而只能检查时间戳、间隔和半速/倍速候选，不能据此判定输出准确，也不能由某个输出反推打击乐片段属于散板或自由节奏。


## 6. 学习型方法：Beat This!（可选）

Beat This! 使用 22.05 kHz 音频、128-bin mel、441-sample hop，交替使用卷积及频率/时间 Transformer，模型约 20M 参数，并以轻量峰值后处理替代 DBN。论文在其公开数据集评测中报告了有竞争力的 F1 结果，同时指出 continuity 指标并非总更好，困难或代表不足的流派仍会失败。

下面比较无真值条件下的调用结果。若包未预先安装则明确跳过，不在 Notebook 中自动安装。


In [ ]:
beat_this_available = False
try:
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning, module="rotary_embedding_torch")
    from beat_this.inference import File2Beats
    beat_this_available = True
    print("Beat This! 已安装")
except ImportError:
    print("Beat This! 未安装。如需对比，请运行：")
    print("  pip install beat-this")

In [ ]:
if beat_this_available:
    import soundfile as sf
    temp_path = OUTPUT_AUDIO_DIR / "_temp_beat_this_input.wav"
    sf.write(temp_path, song_samples, SAMPLE_RATE, subtype="PCM_16")
    try:
        file2beats = File2Beats(device="cpu")
        beat_this_beats, beat_this_downbeats = file2beats(str(temp_path))

        fig, ax = plt.subplots(figsize=(14, 3))
        librosa.display.waveshow(song_samples, sr=SAMPLE_RATE, ax=ax, alpha=0.5, color="0.7")
        ax.vlines(beat_times_song, -1, 1, color="0.2", ls="--", lw=1.2,
                  label=f"librosa.beat ({len(beat_times_song)})")
        ax.vlines(beat_this_beats, -1, 1, color="0.55", ls="-", lw=1.2,
                  label=f"Beat This! ({len(beat_this_beats)})")
        ax.set_title("无真值节拍输出对比")
        ax.set_xlabel("时间 (s)")
        ax.legend(loc="upper right", ncol=2, frameon=False)
        plt.tight_layout()
        plt.savefig(OUTPUT_FIG_DIR / "beat_this_comparison.png", dpi=600, bbox_inches="tight")
        plt.show()

        print("librosa 前 5 个节拍：", np.round(beat_times_song[:5], 3))
        print("Beat This! 前 5 个节拍：", np.round(np.asarray(beat_this_beats)[:5], 3))
        print(f"Beat This! 小节首拍：{len(beat_this_downbeats)} 个")
    except Exception as exc:
        print(f"Beat This! 推理失败，跳过可选对比：{type(exc).__name__}: {exc}")
    finally:
        temp_path.unlink(missing_ok=True)
else:
    print("跳过 Beat This! 对比（运行前未安装）。")


## 7. 小结

1. `librosa.feature.tempogram` 是局部自相关；傅里叶 tempogram 是另一个函数。
2. `beat_track` 的速度估计含先验，动态规划使用相对周期的对数惩罚。
3. Beat This! 无 DBN，但 F1、continuity 与不同流派上的表现不能混为一个结论。
4. 当前两段录音没有节拍真值，所有输出对比都限定为定性诊断。


In [ ]:
print("本 Notebook 生成的图像文件：")
for prefix in ["onset_", "tempogram", "beat_tracking", "beat_this"]:
    for f in OUTPUT_FIG_DIR.glob(f"{prefix}*.png"):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s} {size_kb:8.1f} KB")